# Antigravity Agent - Ambiente de Execução Unificado
Este notebook instala o servidor do agente, mapeia seu Google Drive e expõe na web em uma única célula para evitar erros de diretório.

In [ ]:
NGROK_TOKEN = "" # @param {type:"string"}
GEMINI_API_KEY = "" # @param {type:"string"}

import os
import sys
import subprocess
import time
from google.colab import drive

# 0. Configurar Chaves de API
if NGROK_TOKEN:
    os.environ["NGROK_AUTHTOKEN"] = NGROK_TOKEN
if GEMINI_API_KEY:
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

# 1. Garante que estamos na pasta certa
os.chdir('/content')

# 2. Conectar o Google Drive
drive.mount('/content/drive')
DRIVE_AGENT_DIR = "/content/drive/MyDrive/Antigravity_Agent/Drive"
os.makedirs(DRIVE_AGENT_DIR, exist_ok=True)
print("✅ 1. Google Drive conectado!")

# 3. Baixar o Código e Preparar a Ponte
REPO_URL = "https://github.com/fabioc02/Antigravity-Agent"
os.system('rm -rf /content/applet')
os.system(f'git clone {REPO_URL} /content/applet')

# Sincronizar pastas de estado/memória para o Google Drive
os.system(f'cp -rn /content/applet/Drive/* {DRIVE_AGENT_DIR}/ 2>/dev/null')

# Criar Symlink
os.system('rm -rf /content/applet/Drive')
os.system(f'ln -s {DRIVE_AGENT_DIR} /content/applet/Drive')
print("✅ 2. Repositório clonado e memória sincronizada!")

# 4. Instalar Dependências (Ocultando logs longos)
os.chdir('/content/applet')
print("⏳ Instalando dependências (Node, Python e Ngrok)... Isso pode demorar 1-2 minutos.")
os.system('apt-get update > /dev/null 2>&1')
os.system('apt-get install -y lsof > /dev/null 2>&1')
os.system('curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1')
os.system('apt-get install -y nodejs > /dev/null 2>&1')
os.system('npm install > /dev/null 2>&1')
os.system(f'{sys.executable} -m pip install -r requirements.txt pyngrok > /dev/null 2>&1')
print("✅ 3. Dependências instaladas!")

# 5. MATAR APENAS AS PORTAS DO APP E NGROK
os.system('lsof -ti:3000 | xargs kill -9 2>/dev/null')
os.system('lsof -ti:8082 | xargs kill -9 2>/dev/null')
os.system('pkill -f ngrok')

# 6. Iniciar Servidores (Detecção automática de Hardware: Qwen GPU vLLM ou Qwen CPU Transformers)
print("⏳ Iniciando servidores (Se estiver sem GPU, conectará como Chatbot lento na CPU)...")
env = os.environ.copy()
env['PYTHON_EXEC'] = sys.executable
subprocess.Popen(["npm", "run", "dev"], cwd="/content/applet", env=env, stdout=open('/content/applet/server_out.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(15)

# 7. Expor na Web com Ngrok
from pyngrok import ngrok
if not NGROK_TOKEN:
    print("\n❌ ERRO CRÍTICO: Você não colocou o NGROK_TOKEN na primeira linha da célula! O link não será gerado.")
else:
    ngrok.set_auth_token(NGROK_TOKEN)
    public_url = ngrok.connect(3000).public_url
    print("\n" + "="*60)
    print("🌍 ACESSE SEU AGENTE PELO LINK ABAIXO:")
    print(f"👉  {public_url}  👈")
    print("="*60 + "\n")

# 8. Mostrar eventuais erros caso não tenha ligado corretamente
time.sleep(2)
os.system('tail -n 15 /content/applet/server_out.log')
